# Default notebook

This default notebook is executed using a Lakeflow job as defined in resources/sample_job.job.yml.

In [0]:
%run ./utils/logger

In [0]:
run_id = get_run_id()
print(run_id)

In [0]:
# Set default catalog and schema
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"USE SCHEMA {schema}")

In [0]:
orderitems_df = spark.sql(f"""
SELECT *, (_metadata.file_name) as file_name
FROM read_files(
    'abfss://ecommerce@dataprojectadls.dfs.core.windows.net/{catalog}/inbound/order_items*.csv',
    format => 'csv',
    inferSchema => true
)
""")

In [0]:
display(orderitems_df)

In [0]:
files_recieved = orderitems_df.count()

if files_recieved > 0:
    print(f'Number of files recieved :{files_recieved}')
else:
    print('No files present in adls path')

In [0]:
try:
    spark.sql(f""" create table if not exists {catalog}.{schema}.orderitems_stage
              as
              SELECT *, (_metadata.file_name) as file_name
              FROM read_files(
                  'abfss://ecommerce@dataprojectadls.dfs.core.windows.net/{catalog}/inbound/order_items*.csv',
                  format => 'csv',
                  inferSchema => true
                  )
                  """)

    log_run(run_id, "orderitems_ingest_pipeline", "orderitems_raw", "SUCCESS", "order items raw data satge load completed")

except Exception as e:
    log_run(run_id, "orderitems_ingest_pipeline", "orderitems_raw", "FAILED", error_message=str(e))
    raise 

In [0]:
%sql
describe table extended orderitems_stage

In [0]:
%sql
describe table extended order_items

In [0]:
try:

    spark.sql(f"""
        TRUNCATE TABLE {catalog}.bronze.order_items
    """)

    spark.sql(f"""
        INSERT INTO {catalog}.bronze.order_items
        SELECT
            CAST(order_id AS STRING)               AS order_id,
            CAST(product_id AS STRING)             AS product_id,
            CAST(quantity AS INT)                  AS quantity,
            CAST(unit_price AS DECIMAL(10,2))      AS unit_price,
            CURRENT_TIMESTAMP()                    AS ingestion_time,
            file_name                              AS source_file
        FROM {catalog}.bronze.orderitems_stage
    """)

    log_run(
        run_id,
        "order_items_ingest_pipeline",
        "bronze_load",
        "SUCCESS",
        "Order items bronze load completed"
    )

except Exception as e:

    log_run(
        run_id,
        "order_items_ingest_pipeline",
        "bronze_load",
        "FAILED",
        error_message=str(e)
    )

    raise

In [0]:
spark.sql(f'DROP TABLE {catalog}.{schema}.orderitems_stage');